# OpenTelemetry Load

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

# Starter Code Import

In [2]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"
!wget {PREFIX}/rag_helper.py
!wget {PREFIX}/starter.py

--2026-07-20 16:32:46--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1814 (1.8K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   1.77K  --.-KB/s    in 0s      

2026-07-20 16:32:46 (5.73 MB/s) - ‘rag_helper.py’ saved [1814/1814]

--2026-07-20 16:32:46--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/starter.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting res

# Library + Starter Code Load

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from rag_helper import RAGBase, INSTRUCTIONS, PROMPT_TEMPLATE
from starter import index, client

class RAGTraced(RAGBase):

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search_operation") as span:
            return self.index.search(query, num_results=num_results)
        

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append(doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        with tracer.start_as_current_span("llm_operation") as span:
            input_messages = [
                {'role': 'developer', 'content': self.instructions},
                {'role': 'user', 'content': prompt}
            ]

            response = self.llm_client.responses.create(
                model=self.model,
                input=input_messages
            )

            input_cost = response.usage.input_tokens * (0.75 / 1_000_000)
            output_cost = response.usage.output_tokens * (4.50 / 1_000_000)
            cost = input_cost + output_cost

            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            span.set_attribute("cost", cost)
            
            return response
        
    

    def rag(self, query):
        with tracer.start_as_current_span("rag_operation") as span:
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            return response.output_text
    


In [5]:
rag_traced = RAGTraced(index=index, llm_client=client)

## Q1. First trace
Wrap the `rag()` method so each call produces a span. The simplest way
is to create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`,
`search()`, and `llm()` each in their own span.

Run this query:

> How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary.
Count the spans in the console output - each one is a separate
`ReadableSpan` entry. How many spans does the trace produce?

* 1
* 3
* 5
* 7

In [6]:
answer = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0xdc986a2b3a54704f68bac21e6023ca20",
        "span_id": "0x1517ec424d63e645",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xbae6bd486bbf203e",
    "start_time": "2026-07-20T16:33:00.355003Z",
    "end_time": "2026-07-20T16:33:00.358387Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "db0b06b4-b9b6-456a-84e6-f886db608257",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0xdc986a2b3a54704f68bac21e6023ca20",
        "span_id": "0xd01bc9174d8d6285",
        "trace_state": "[]"
    },
 

## Q1 Answer: 3
Because we defined this for 3 items: `search_operation`, `llm_operation`, and `rag_operation`

## Q2. Capturing metrics as span attributes
Spans are not just timing markers - you can attach any information you
want to them with `set_attribute`. We already use spans to record how
long each step takes. Now we'll add the metrics we care about: tokens
and cost.

Read the token usage from the LLM response (the `llm()` method in the
starter already returns the raw response object) and set them as
attributes on the `llm` span:

```
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute
the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

* 700
* 7000
* 70000
* 700000

## Q2 answer: "input_tokens": 7111, so ~ 7000

## Q3: Span timing

Each span automatically records its duration. Look at the console output
from Q1 and find the durations for the `search` span and the `llm` span.

For a typical query, roughly how long does the LLM call take?

* Under 100ms
* 100-500ms
* 500-2000ms
* Over 2000ms

> The first call can be slower (cold start). Pick the range you see
> most often.

search operation -     "start_time": "2026-07-20T16:33:00.355003Z", "end_time": "2026-07-20T16:33:00.358387Z",
<br>llm operation -      "start_time": "2026-07-20T16:33:00.360697Z", "end_time": "2026-07-20T16:33:02.927407Z",

In [8]:
from datetime import datetime

# Define the timestamps and their format
time_format = "%Y-%m-%dT%H:%M:%S.%fZ"



search_start_time_str = "2026-07-20T16:33:00.355003Z"
search_end_time_str = "2026-07-20T16:33:00.358387Z"

llm_start_time_str = "2026-07-20T16:33:00.360697Z"
llm_end_time_str = "2026-07-20T16:33:02.927407Z"

# Convert strings to datetime objects
search_start_time = datetime.strptime(search_start_time_str, time_format)
search_end_time = datetime.strptime(search_end_time_str, time_format)

llm_start_time = datetime.strptime(llm_start_time_str, time_format)
llm_end_time = datetime.strptime(llm_end_time_str, time_format)

search_time_diff = search_end_time - search_start_time
llm_time_diff = llm_end_time - llm_start_time
total_time_diff = search_time_diff + llm_time_diff

print(f"Search duration: {search_time_diff.total_seconds() * 1000:.2f}ms")
print(f"LLM duration: {llm_time_diff.total_seconds() * 1000:.2f}ms")
print(f"Total duration: {total_time_diff.total_seconds() * 1000:.2f}ms")

Search duration: 3.38ms
LLM duration: 2566.71ms
Total duration: 2570.09ms


In [9]:
answer = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0x76679b71d4026af821b47f69a5c12d38",
        "span_id": "0x7b602d2783389672",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xbe730e0e1a9c2b1a",
    "start_time": "2026-07-20T16:35:37.533650Z",
    "end_time": "2026-07-20T16:35:37.538517Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "db0b06b4-b9b6-456a-84e6-f886db608257",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0x76679b71d4026af821b47f69a5c12d38",
        "span_id": "0x0855825a5c6bce2e",
        "trace_state": "[]"
    },
 

In [10]:
from datetime import datetime

# Define the timestamps and their format
time_format = "%Y-%m-%dT%H:%M:%S.%fZ"

search_start_time_str = "2026-07-20T16:35:37.533650Z"
search_end_time_str = "2026-07-20T16:35:37.538517Z"

llm_start_time_str = "2026-07-20T16:35:37.540586Z"
llm_end_time_str = "2026-07-20T16:35:39.077295Z"

# Convert strings to datetime objects
search_start_time = datetime.strptime(search_start_time_str, time_format)
search_end_time = datetime.strptime(search_end_time_str, time_format)

llm_start_time = datetime.strptime(llm_start_time_str, time_format)
llm_end_time = datetime.strptime(llm_end_time_str, time_format)

search_time_diff = search_end_time - search_start_time
llm_time_diff = llm_end_time - llm_start_time
total_time_diff = search_time_diff + llm_time_diff

print(f"Search duration: {search_time_diff.total_seconds() * 1000:.2f}ms")
print(f"LLM duration: {llm_time_diff.total_seconds() * 1000:.2f}ms")
print(f"Total duration: {total_time_diff.total_seconds() * 1000:.2f}ms")

Search duration: 4.87ms
LLM duration: 1536.71ms
Total duration: 1541.58ms


## Q3 Answer: the response for the second run is 1536 ms, so 500-2000ms